# LLM JSON Schema — Structured Output

Extract structured metadata from a real academic paper PDF, constraining each provider's output to the same JSON schema so the response always parses cleanly to a Python dict — no regex, no hallucinated keys.

Each provider has its own mechanism; all share the same schema and prompt.

In [ ]:
import json
import os
from pypdf import PdfReader

with open('paper_schema.json') as f:
    SCHEMA = json.load(f)

reader = PdfReader('Towards_a_Platform_for_AI_Assisted_Papyrology.pdf')
pdf_text = '\n'.join(page.extract_text() for page in reader.pages)

SCHEMA_PROMPT = f"""
Extract structured metadata from the following academic paper:

{pdf_text}
"""

---
## OpenAI

Passed as `response_format` with `type='json_schema'`. The SDK validates the response automatically when `strict=True`.

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL   = 'gpt-5'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
openai_resp = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{'role': 'user', 'content': SCHEMA_PROMPT}],
    response_format={
        'type': 'json_schema',
        'json_schema': {
            'name': 'book_recommendation',
            'strict': True,
            'schema': SCHEMA
        }
    }
)
result = json.loads(openai_resp.choices[0].message.content)
print(json.dumps(result, indent=2))

---
## Anthropic

Uses `output_config` to attach the JSON schema directly to the request. The SDK also offers `client.messages.parse()` for automatic Pydantic model validation.

In [ ]:
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL   = 'claude-sonnet-4-5-20250929'

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
anthropic_resp = anthropic_client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=500,
    messages=[{'role': 'user', 'content': SCHEMA_PROMPT}],
    output_config={'format': {'type': 'json_schema', 'schema': SCHEMA}}
)
result = json.loads(anthropic_resp.content[0].text)
print(json.dumps(result, indent=2))

---
## Google Gemini

Set `response_mime_type='application/json'` in `GenerationConfig` and pass the schema in the prompt so Gemini knows the exact shape.

In [ ]:
import google.generativeai as genai

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GOOGLE_MODEL   = 'gemini-2.5-flash'

genai.configure(api_key=GEMINI_API_KEY)
generation_config = genai.GenerationConfig(response_mime_type='application/json')
gemini_schema_model = genai.GenerativeModel(GOOGLE_MODEL, generation_config=generation_config)

schema_str = json.dumps(SCHEMA, indent=2)
gemini_resp = gemini_schema_model.generate_content(
    f'{SCHEMA_PROMPT}\n\nRespond as JSON matching this schema:\n{schema_str}'
)
result = json.loads(gemini_resp.text)
print(json.dumps(result, indent=2))

---
## Ollama (local)

Pass the JSON schema directly to the `format=` parameter. Ollama constrains the tokenizer at the grammar level — the output is always valid JSON.

In [ ]:
import ollama

OLLAMA_MODEL = 'mistral-nemo:12b-instruct-2407-q4_K_M'

ollama_resp = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[{'role': 'user', 'content': SCHEMA_PROMPT}],
    format=SCHEMA,
    options={'temperature': 0.2}
)
result = json.loads(ollama_resp['message']['content'])
print(json.dumps(result, indent=2))